# Hibiki-Sw: Data Preparation (Tokenizer + Text + Audio Encoding)

**GPU Budget: ~2 hours** (encoding audio through Mimi codec)

This notebook prepares the non-speech-generation artifacts for the training pipeline:
1. Train SentencePiece tokenizer (CPU)
2. Tokenize text data for Stage 1 (CPU)
3. Encode audio to Mimi codec tokens for Stage 2 (GPU)
4. Create S2ST manifests for Stages 3-4 (CPU)

**Prerequisites:**
- Common Voice dataset downloaded locally (from Mozilla Data Collective)
- For S2ST manifests: run `00a_stage0_vits.ipynb` + `00b_data_pipeline.ipynb` first

**Output artifacts** are saved to Google Drive / Kaggle Dataset for reuse across sessions.

In [ ]:
# Install dependencies
!pip install -q transformers accelerate sentencepiece datasets soundfile scipy torchaudio faster-whisper sacrebleu

In [ ]:
import os
import sys

# Clone or upload your repo
# !git clone https://github.com/gambigivesyouwiings/Mambo.git /kaggle/working/hibiki-sw
# Or if uploaded as a Kaggle dataset:
REPO_DIR = "/kaggle/input/hibiki-sw-code/hibiki-sw"  # adjust path
sys.path.insert(0, REPO_DIR)

OUTPUT_DIR = "/kaggle/working/data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Common Voice dataset paths (adjust to your extracted paths)
# On Kaggle, upload the dataset or mount from a Kaggle Dataset
CV_SW_DIR = "/kaggle/input/cv-swahili/cv-corpus-19.0-2024-09-13/sw"  # ADJUST
CV_EN_DIR = "/kaggle/input/cv-english/cv-corpus-19.0-2024-09-13/en"  # ADJUST (optional)

print(f"Repo: {REPO_DIR}")
print(f"Output: {OUTPUT_DIR}")
print(f"CV Swahili: {CV_SW_DIR}")
print(f"CV English: {CV_EN_DIR}")

## 1. Train SentencePiece Tokenizer (~10 min, CPU)

In [ ]:
TOKENIZER_DIR = f"{OUTPUT_DIR}/tokenizer"
os.makedirs(TOKENIZER_DIR, exist_ok=True)

# Train tokenizer using local Common Voice data
# --cv_dataset_dir can be specified multiple times for multiple languages
cmd = f"""python {REPO_DIR}/data/prepare/train_tokenizer.py \
    --output_dir {TOKENIZER_DIR} \
    --vocab_size 32000 \
    --num_sentences 2000000 \
    --cv_dataset_dir sw:{CV_SW_DIR} \
    --cv_split validated"""

# Add English CV if available
if os.path.exists(CV_EN_DIR):
    cmd += f" \\\n    --cv_dataset_dir en:{CV_EN_DIR}"

print(f"Running: {cmd}\n")
!{cmd}

In [ ]:
# Verify tokenizer
import sentencepiece as spm
sp = spm.SentencePieceProcessor()
sp.load(f"{TOKENIZER_DIR}/sp_ensw_32k.model")
print(f"Vocab size: {sp.get_piece_size()}")
print(f"EN test: {sp.encode('Hello, how are you?', out_type=str)}")
print(f"SW test: {sp.encode('Habari, hali yako?', out_type=str)}")

## 2. Tokenize Text for Stage 1 (~5 min, CPU)

In [ ]:
TEXT_TOKEN_DIR = f"{OUTPUT_DIR}/text_tokens"

# Tokenize text using local Common Voice data
cmd = f"""python {REPO_DIR}/data/prepare/tokenize_text.py \
    --tokenizer_model {TOKENIZER_DIR}/sp_ensw_32k.model \
    --output_dir {TEXT_TOKEN_DIR} \
    --seq_length 1024 \
    --max_samples 500000 \
    --cv_dataset_dir sw:{CV_SW_DIR} \
    --cv_split validated"""

if os.path.exists(CV_EN_DIR):
    cmd += f" \\\n    --cv_dataset_dir en:{CV_EN_DIR}"

print(f"Running: {cmd}\n")
!{cmd}

import glob
n_files = len(glob.glob(f"{TEXT_TOKEN_DIR}/*.npy"))
print(f"Created {n_files} text token files")

## 3. Encode Audio with Mimi Codec (~1.5 hours, GPU)

This is the GPU-intensive step. We encode:
- Common Voice English (~50k samples)
- Common Voice Swahili (~5k samples)
- FLEURS en_us + sw_ke (for S2ST training & eval)

In [ ]:
AUDIO_TOKEN_DIR = f"{OUTPUT_DIR}/audio_tokens"

# Common Voice Swahili (using local dataset)
!python {REPO_DIR}/data/prepare/encode_audio.py \
    --source common_voice --lang sw \
    --split validated \
    --dataset_dir {CV_SW_DIR} \
    --output_dir {AUDIO_TOKEN_DIR}/cv_sw \
    --num_codebooks 8 \
    --max_samples 10000 \
    --max_duration 20.0

In [ ]:
# Common Voice English (using local dataset, if available)
if os.path.exists(CV_EN_DIR):
    !python {REPO_DIR}/data/prepare/encode_audio.py \
        --source common_voice --lang en \
        --split validated \
        --dataset_dir {CV_EN_DIR} \
        --output_dir {AUDIO_TOKEN_DIR}/cv_en \
        --num_codebooks 8 \
        --max_samples 50000 \
        --max_duration 20.0
else:
    print("English CV not available locally, skipping.")

In [ ]:
# FLEURS en_us + sw_ke (train + test)
for lang in ["en_us", "sw_ke"]:
    for split in ["train", "test"]:
        print(f"\nEncoding FLEURS {lang} {split}...")
        !python {REPO_DIR}/data/prepare/encode_audio.py \
            --source fleurs --lang {lang} --split {split} \
            --output_dir {AUDIO_TOKEN_DIR}/fleurs_{lang}

## 4. Create S2ST Manifests (~5 min, CPU)

In [ ]:
MANIFEST_DIR = f"{OUTPUT_DIR}/manifests"
os.makedirs(MANIFEST_DIR, exist_ok=True)

# FLEURS en->sw manifest
!python {REPO_DIR}/data/prepare/create_s2st_manifest.py \
    --source fleurs \
    --source_token_dir {AUDIO_TOKEN_DIR}/fleurs_en_us \
    --target_token_dir {AUDIO_TOKEN_DIR}/fleurs_sw_ke \
    --text_token_dir {OUTPUT_DIR}/aligned_text/en2sw \
    --output_manifest {MANIFEST_DIR}/fleurs_en2sw_train.tsv \
    --tokenizer_model {TOKENIZER_DIR}/sp_ensw_32k.model \
    --direction en2sw

## 5. Save as Kaggle Dataset

Upload the prepared data so it persists across notebook sessions.

In [ ]:
# Check total data size
!du -sh {OUTPUT_DIR}/*
!echo "Total:"
!du -sh {OUTPUT_DIR}

In [ ]:
# Create dataset metadata for Kaggle
import json

metadata = {
    "title": "hibiki-sw-data",
    "id": "YOUR_USERNAME/hibiki-sw-data",  # CHANGE THIS
    "licenses": [{"name": "CC-BY-4.0"}]
}

with open(f"{OUTPUT_DIR}/dataset-metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

# Upload (requires Kaggle API key)
# !kaggle datasets create -p {OUTPUT_DIR}
print("Data preparation complete! Upload to Kaggle Datasets for persistence.")